# Homework 4: SARSA, Expected SARSA, Q-learning and the Cross-Entropy Method on Taxi

**ESE 561, Theory of Artificial Intelligence. 20 points.** Submit this notebook **executed**, with every written answer in the cells marked *Your answer*.

All the code is written for you, except the small task in Q1. Your job is to **read** the code, **run** it, look at the plots and animations, and **explain** what you see. Several questions ask you to change a parameter and rerun a cell: do it.

| Part | Topic | Points |
|---|---|---|
| 1 | The environment (Q1); TD control: SARSA, Expected SARSA, Q-learning (Q1.1–Q1.3) | 8 |
| 2 | Tabular cross-entropy method (Q2.1–Q2.2) | 5 |
| 3 | Theory (T1–T5) | 5 |
| 4 | Planning versus learning: value iteration (Q4) | 2 |

Everything runs on a laptop CPU in a few minutes.

In [ ]:
# !pip install "gymnasium[toy-text]"      # uncomment if needed
import time
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

env = gym.make("Taxi-v4", render_mode="rgb_array") #rgb_array
nS, nA = env.observation_space.n, env.action_space.n
print(nS, 'states,', nA, 'actions')
s, _ = env.reset(seed=0)
#print(env.render())
print('state', s, '= (taxi row, taxi col, passenger location, destination):', list(env.unwrapped.decode(s)))
plt.imshow(env.render())

## The environment

A $5\times5$ grid with walls (`|`) and four stands R, G, Y, B. The passenger waits at one stand and wants to go to another.
* **State** (500): taxi row and column (25), passenger location (one of the 4 stands, or 4 = in the taxi), destination (4).
* **Actions** (6): 0 south, 1 north, 2 east, 3 west, 4 pickup, 5 drop-off.
* **Rewards:** $-1$ per step, $+20$ for delivering the passenger (the episode **terminates**), $-10$ for an illegal pickup or drop-off.
* **Time limit:** the episode is **truncated** after 200 steps.

The dynamics are deterministic; only the starting state is random: `env.reset(seed=k)` picks the taxi position, the passenger's stand and the destination (always different from the passenger's stand) from the random number generator with seed `k`. The same seed always gives the same start, which is how we compare policies on the same problems below.

### Q1 (1 pt). Try a different random start
By hand, find a shortest sequence of actions that delivers the passenger (mind the walls: the taxi cannot cross a `|`). How many steps does it take, and what return does it get? Every move and the pickup cost $-1$; the final drop-off gives $+20$ instead.

In Part 4 you will check your answer against the optimal policy computed by value iteration.

In [2]:
MY_SEED = ...   # any integer except 0
#Your code is here ╰( ͡° ͜ʖ ͡° )つ──☆*:


*Your answer:*

&nbsp;

A random policy:

In [ ]:
rng = np.random.default_rng(0)
R_random = []
for k in range(200):
    s, _ = env.reset(seed=k)
    G, done = 0, False
    while not done:
        s, r, term, trunc, _ = env.step(int(rng.integers(nA)))
        G += r; done = term or trunc
    R_random.append(G)
print(f'random policy: mean return {np.mean(R_random):.1f}  (an optimal policy gets about +8)')

### Watching an episode

`env.render()` returns an RGB image of the current state, so we can record one frame per step and play the episode as an animation. Use the slider and buttons under the animation to step through it. **What you see:** the car is the taxi; the person icon is the passenger, waiting at one of the four coloured stands R, G, Y, B until picked up (then it disappears into the car); the hotel icon marks the destination.

The helpers below take any **policy**, a function `state -> action`, so we can use them for the random policy now and for the learned policies later.

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display
plt.rcParams['animation.embed_limit'] = 50          # MB, so that long animations are not cut

ACTION_NAMES = ['south', 'north', 'east', 'west', 'pickup', 'dropoff']

def record_episode(policy, seed=0, max_steps=200):
    """Play one episode with policy(s) -> a from the start state given by `seed`.
    Returns frames (one more than steps), actions and rewards."""
    s, _ = env.reset(seed=seed)
    frames, actions, rewards, done = [env.render()], [], [], False
    while not done and len(actions) < max_steps:
        a = int(policy(s))
        s, r, term, trunc, _ = env.step(a)
        frames.append(env.render()); actions.append(a); rewards.append(r); done = term or trunc
    return frames, actions, rewards

def _caption(name, actions, rewards, t):
    G = sum(rewards[:t])
    last = f'{ACTION_NAMES[actions[t - 1]]}, r = {rewards[t - 1]:+d}' if t > 0 else 'start'
    return f'{name}\nstep {t}: {last}   return {G:+d}'

def animate_episodes(episodes, max_frames=60, interval=350):
    """episodes: list of (name, (frames, actions, rewards)), shown side by side.
    Shows at most max_frames steps; a finished episode keeps showing its last frame."""
    n = min(max(len(ep[1][0]) for ep in episodes), max_frames + 1)
    fig, axes = plt.subplots(1, len(episodes), figsize=(4.2 * len(episodes), 3.6), squeeze=False)
    ims = []
    for ax, (name, (fr, ac, rw)) in zip(axes[0], episodes):
        ims.append(ax.imshow(fr[0])); ax.axis('off')
    def update(t):
        for ax, im, (name, (fr, ac, rw)) in zip(axes[0], ims, episodes):
            k = min(t, len(fr) - 1)
            im.set_data(fr[k]); ax.set_title(_caption(name, ac, rw, k), fontsize=9)
        return ims
    update(0); plt.tight_layout()
    anim = animation.FuncAnimation(fig, update, frames=n, interval=interval)
    plt.close(fig)
    for name, (fr, ac, rw) in episodes:
        more = f'  (animation shows the first {max_frames})' if len(ac) > max_frames else ''
        print(f'{name}: {len(ac)} steps, return {sum(rw):+d}{more}')
    return HTML(anim.to_jshtml())

def filmstrip(name, episode, n=12, cols=6):
    """A static strip of n frames spread over the episode (visible in the PDF/HTML of the notebook too)."""
    fr, ac, rw = episode
    idx = np.unique(np.linspace(0, len(fr) - 1, min(n, len(fr))).astype(int))
    rows = int(np.ceil(len(idx) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 1.9 * rows), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, t in zip(axes.flat, idx):
        ax.imshow(fr[t]); ax.set_title(f't={t}' + (f' {ACTION_NAMES[ac[t - 1]]}' if t else ''), fontsize=8)
    fig.suptitle(f'{name}: {len(ac)} steps, return {sum(rw):+d}'); plt.tight_layout(); plt.show()

In [ ]:
rng_play = np.random.default_rng(1)
ep_random = record_episode(lambda s: rng_play.integers(nA), seed=0)
display(animate_episodes([('random policy', ep_random)], max_frames=100))
filmstrip('random policy', ep_random)

The random taxi wanders, tries illegal pickups and drop-offs ($-10$ each), and almost never delivers the passenger before the 200-step limit.

## Part 1 (continued). TD control

All three methods share one loop. After each step $(s,a)\to(r,s')$ the agent samples its next action $a'$ from the ε-greedy policy and moves $Q(s,a)$ towards a target $Y$:
$$
Q(s,a)\leftarrow Q(s,a)+\alpha\,\bigl(Y-Q(s,a)\bigr),
$$
| method | target $Y$ (if $s'$ is terminal: $Y=r$)__________ |
|---|---|
| SARSA | $r+\gamma\,Q(s',a')$ |
| Expected SARSA | $r+\gamma\sum_b\pi_\varepsilon(b\mid s')\,Q(s',b)$ |
| Q-learning | $r+\gamma\max_bQ(s',b)$ |

The Q-table is a numpy array `Q[s, a]` of shape $500\times6$.

**The ε-greedy policy.** For one row `q = Q[s]`, every action gets probability $\varepsilon/n$ ($n=6$ actions) and the greedy action $\arg\max_a q_a$ gets an extra $1-\varepsilon$. Example: `q = [1, 3, 2]`, $\varepsilon=0.3$, $n=3$ gives `[0.1, 0.8, 0.1]`. This is the same as "with probability ε pick a uniformly random action, otherwise the greedy one"; note that the greedy action is then chosen with probability $1-\varepsilon+\varepsilon/n$, because the random choice can also hit it. We need the probabilities explicitly for Expected SARSA, which averages over them.

In [ ]:
def eps_greedy_probs(q, eps):
    # eps/n for every action, plus (1 - eps) for the greedy one
    p = np.full(len(q), eps / len(q)); p[np.argmax(q)] += 1 - eps
    return p

def eps_greedy_action(Q, s, eps, rng):
    # sample one action from the eps-greedy probabilities of state s
    return int(rng.choice(len(Q[s]), p=eps_greedy_probs(Q[s], eps)))

**The three targets** for a **non-terminal** $s'$ (the loop below handles the terminal case). A worked example: $r=-1$, $\gamma=0.9$, $\varepsilon=0.3$, $Q(s',\cdot)=[1,4,2]$, and the sampled next action is $a'=2$:

| method | target | value |
|---|---|---|
| SARSA | $r+\gamma Q(s',a')=-1+0.9\cdot2$ | 0.80 |
| Expected SARSA | $r+\gamma(0.1\cdot1+0.8\cdot4+0.1\cdot2)$ | 2.15 |
| Q-learning | $r+\gamma\max_bQ(s',b)=-1+0.9\cdot4$ | 2.60 |

SARSA uses the action the agent will **actually** take next; here exploration sampled a poor action, so the target is low, and another time it would be high: the target is random. Expected SARSA averages over what the ε-greedy policy **might** do: the same value every time. Q-learning assumes the **best** action will be taken next, whatever the agent actually does: it learns about the greedy policy while following the ε-greedy one (off-policy).

In [ ]:
def sarsa_target(r, s2, a2, Q, gamma, eps):
    return r + gamma * Q[s2, a2]

def expected_sarsa_target(r, s2, a2, Q, gamma, eps):
    return r + gamma * eps_greedy_probs(Q[s2], eps) @ Q[s2]

def q_learning_target(r, s2, a2, Q, gamma, eps):
    return r + gamma * Q[s2].max()

**The training loop.** One function trains any of the three methods: the only difference is the `target` function. Read it line by line: reset, choose $a$; then repeat: step, choose $a'$, build the target $Y$, move $Q(s,a)$ a fraction α towards $Y$, shift $(s,a)\leftarrow(s',a')$.

**Bootstrapping.** All three targets contain $Q(s',\cdot)$: our **own current estimate** of the value of the next state stands in for the true future return, which we would only know at the end of the episode. Updating an estimate from another estimate is called **bootstrapping** in RL, after "pulling yourself up by your own bootstraps". Monte Carlo methods (random actions), and CEM in Part 2, do **not** bootstrap: they wait until the episode ends and use the return that actually happened. Bootstrapping lets TD methods learn after every single step, and passes information backwards one step per update; the price is that early targets are built on wrong estimates. This is **not** the bootstrap of statistics (Efron's bootstrap: resampling the data with replacement to estimate the variability of a statistic); only the metaphor is shared.

**Terminated versus truncated.** Gymnasium returns two flags, and the loop treats them differently:
* `terminated`: the passenger was delivered. The task is over, nothing more can happen, the value of what follows is exactly 0, so $Y=r$: no bootstrapping.
* `truncated`: the 200-step limit was hit. The task is **not** over; the taxi is somewhere with the passenger still waiting, we simply stopped watching. The future value is not 0, so the target **still bootstraps**: $Y=r+\gamma\max_bQ(s',b)$ (or the SARSA / Expected SARSA version).

Both flags end the episode (`done = term or trunc`), but only `term` cuts the target. Theory question T3 asks what goes wrong otherwise.

`greedy_return` evaluates a Q-table: it plays $\arg\max_aQ(s,a)$ without exploration from 50 fixed start states and averages the returns.

In [ ]:
TARGETS = {'SARSA': sarsa_target, 'Expected SARSA': expected_sarsa_target, 'Q-learning': q_learning_target}

def td_control(method, n_episodes=2000, alpha=0.5, eps=0.1, gamma=0.99, glie=False, seed=0):
    """Returns (returns per episode, Q). glie=True uses eps_k = eps / (1 + k/100) in episode k."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((nS, nA)); returns = []
    target = TARGETS[method]
    for ep in range(n_episodes):
        e = eps / (1 + ep / 100) if glie else eps
        s, _ = env.reset(seed=int(rng.integers(1 << 31)))
        a = eps_greedy_action(Q, s, e, rng)
        G, done = 0.0, False
        while not done:
            s2, r, term, trunc, _ = env.step(a); G += r; done = term or trunc
            a2 = eps_greedy_action(Q, s2, e, rng)               # the next action (SARSA uses it in the target)
            Y = r if term else target(r, s2, a2, Q, gamma, e)   # bootstrap unless terminated (not: truncated)
            Q[s, a] += alpha * (Y - Q[s, a])                    # move a fraction alpha towards the target
            s, a = s2, a2
        returns.append(G)
    return np.array(returns), Q

def greedy_return(Q, n=50, seed=12345):
    """Average return of the greedy policy (eps = 0) over n fixed start states."""
    G = []
    for k in range(n):
        s, _ = env.reset(seed=seed + k); g, done = 0.0, False
        while not done:
            s, r, term, trunc, _ = env.step(int(np.argmax(Q[s]))); g += r; done = term or trunc
        G.append(g)
    return np.mean(G)

### Q1.1 (3 pt). Three methods, two ways of measuring

Run each method for 2000 episodes, 5 seeds, $\alpha=0.5$, $\varepsilon=0.1$, $\gamma=0.99$. We record two things:
* the return of each training episode, **while exploring** (ε-greedy);
* the return of the **greedy** policy $\arg\max_aQ(s,a)$ after training, on 50 fixed start states.

In [ ]:
def smooth(x, w=50):
    return np.convolve(x, np.ones(w) / w, mode='valid')

t0 = time.time(); runs = {}
for m in TARGETS:
    runs[m] = [td_control(m, seed=k) for k in range(5)]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for m, out in runs.items():
    curve = np.mean([R for R, _ in out], axis=0)
    ax[0].plot(smooth(curve), label=m); ax[1].plot(smooth(curve), label=m)
ax[0].set(xlabel='episode', ylabel='return (with exploration)', title='learning curves, mean of 5 seeds')
ax[1].set(xlabel='episode', ylim=(-30, 12), title='zoom')
for a in ax: a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f'{"":15s} {"last 200 episodes (exploring)":>30s}   greedy return per seed')
for m, out in runs.items():
    print(f'{m:15s} {np.mean([R[-200:].mean() for R, _ in out]):30.1f}   {[round(float(greedy_return(Q)), 1) for _, Q in out]}')
print(f'({time.time() - t0:.0f} s)')

**Watching the greedy policies.** Below, the greedy policy of Q-learning and of SARSA (seed 0 of each), started from the same state: the evaluation start state on which SARSA's greedy policy does worst. Q-learning drives straight to the passenger and to the destination; SARSA's greedy policy can get stuck going back and forth (a loop) until the time limit.

In [ ]:
Q_ql0, Q_sarsa0 = runs['Q-learning'][0][1], runs['SARSA'][0][1]
greedy = lambda Q: (lambda s: np.argmax(Q[s]))
# the evaluation start state (seed) on which SARSA's greedy policy does worst
worst = min(range(12345, 12345 + 50), key=lambda k: sum(record_episode(greedy(Q_sarsa0), seed=k)[2]))
display(animate_episodes([('Q-learning, greedy', record_episode(greedy(Q_ql0), seed=worst)),
                          ('SARSA, greedy', record_episode(greedy(Q_sarsa0), seed=worst))], max_frames=40))

1. Which methods reach the optimal greedy return (about $+8$)? Why are the returns **during training** lower than the greedy returns for all three?
2. SARSA and Expected SARSA estimate the same function $Q^{\pi_\varepsilon}$, yet their greedy policies can differ a lot. Explain the difference using the targets. (Hint: what is random in each target, given $s,a$? The dynamics of Taxi are deterministic.)

*Your answer:*

&nbsp;

### Q1.2 (2 pt). The step size α

In [ ]:
t0 = time.time()
print(f'{"":12s} {"alpha":>6s} {"last 200 (exploring)":>21s}   greedy return per seed')
for m in ['SARSA', 'Q-learning']:
    for alpha in [0.1, 0.5, 0.9]:
        out = [td_control(m, n_episodes=3000, alpha=alpha, seed=k) for k in range(5)]
        print(f'{m:12s} {alpha:6.1f} {np.mean([R[-200:].mean() for R, _ in out]):21.1f}   {[round(float(greedy_return(Q)), 1) for _, Q in out]}')
print(f'({time.time() - t0:.0f} s)')

Q-learning works even with $\alpha=0.9$; SARSA gets worse as $\alpha$ grows. Explain why, using the Robbins–Monro view of the update (a running mean of the targets). Would Q-learning with $\alpha=0.9$ still work if the taxi sometimes slipped to a random neighbouring cell? (You might need AI to help with the answer.)

*Your answer:*

&nbsp;

### Q1.3 (2 pt). Exploration: ε and GLIE (when we reduce $\varepsilon$)

In [ ]:
t0 = time.time()
print(f'{"":12s} {"exploration":>16s} {"last 200 (exploring)":>21s}   greedy return per seed')
for m in ['SARSA', 'Q-learning']:
    for label, kw in [('eps = 0.01', dict(eps=0.01)), ('eps = 0.3', dict(eps=0.3)), ('GLIE from 0.1', dict(eps=0.1, glie=True))]:
        out = [td_control(m, n_episodes=3000, seed=k, **kw) for k in range(5)]
        print(f'{m:12s} {label:>16s} {np.mean([R[-200:].mean() for R, _ in out]):21.1f}   {[round(float(greedy_return(Q)), 1) for _, Q in out]}')
print(f'({time.time() - t0:.0f} s)')

1. With $\varepsilon=0.3$, compare the two methods' training returns and greedy returns. Which function does each method learn?
2. Why does GLIE ($\varepsilon_k=0.1/(1+k/100)\to0$) fix SARSA? Does this schedule still explore every action infinitely often?
3. Even $\varepsilon=0.01$ works well here. Why is so little exploration enough in Taxi? Give an environment feature that would make it fail.

*Your answer:*

&nbsp;

## Part 2. The tabular cross-entropy method (5 pt)

The policy is a $500\times6$ matrix of probabilities $\pi(a\mid s)$, initially uniform. Each iteration:
1. play $N$ episodes with $\pi$, recording all $(s,a)$ pairs and the return;
2. keep the **elite** episodes: return $\ge$ the `percentile`-th percentile of the $N$ returns;
3. count in the elite episodes $N(s,a)$ = times $a$ was taken in $s$, $N(s)=\sum_aN(s,a)$, and set
$$
\hat\pi(a\mid s)=\frac{N(s,a)}{N(s)}\quad(\text{a state no elite visited keeps its old row}),\qquad
\pi\leftarrow\lambda\,\hat\pi+(1-\lambda)\,\pi .
$$

**The code.** `generate_session` plays one episode with the current policy and records all $(s,a)$ pairs and the return. `select_elites` keeps the episodes whose return is at least the percentile and concatenates their pairs. `update_policy` counts the elite pairs with `np.add.at` (`counts[s, a] += 1` for every pair), divides each row by its total, keeps the old row for states no elite visited, and mixes with the old policy.

A worked example with 3 episodes and percentile 50: the returns are $5,-3,9$, the median is 5, so episodes 1 and 3 are elite. Say state 1 appears twice in them, once with action 0 and once with action 3 (5 actions). Then $\hat\pi(\cdot\mid 1)=[0.5,0,0,0.5,0]$, and with a uniform old row $[0.2,\dots,0.2]$ and $\lambda=0.5$:
$$
\pi(\cdot\mid1)=0.5\,[0.5,0,0,0.5,0]+0.5\,[0.2,0.2,0.2,0.2,0.2]=[0.35,0.1,0.1,0.35,0.1].
$$
Actions the elites never took in state 1 keep probability 0.1: the smoothing keeps exploring (Q2.2).

In [ ]:
def generate_session(pi, rng):
    s, _ = env.reset(seed=int(rng.integers(1 << 31)))
    states, actions, G, done = [], [], 0.0, False
    while not done:
        a = int(rng.choice(nA, p=pi[s]))
        s2, r, term, trunc, _ = env.step(a)
        states.append(s); actions.append(a); G += r; s = s2; done = term or trunc
    return states, actions, G

def select_elites(sessions, percentile):
    returns = [G for _, _, G in sessions]
    threshold = np.percentile(returns, percentile)
    keep = [i for i, G in enumerate(returns) if G >= threshold]
    elite_states = [s for i in keep for s in sessions[i][0]]
    elite_actions = [a for i in keep for a in sessions[i][1]]
    return elite_states, elite_actions, threshold

def update_policy(pi, elite_states, elite_actions, lam):
    counts = np.zeros_like(pi)
    np.add.at(counts, (elite_states, elite_actions), 1)        # counts[s, a] += 1 for every elite pair
    visits = counts.sum(axis=1, keepdims=True)
    new = np.where(visits > 0, counts / np.maximum(visits, 1), pi)
    return lam * new + (1 - lam) * pi

### Q2.1 (2 pt). CEM versus Q-learning

In [ ]:
def cem(n_iter=60, n_sessions=250, percentile=50, lam=0.5, seed=0):
    rng = np.random.default_rng(seed)
    pi = np.full((nS, nA), 1.0 / nA); log = []
    for it in range(n_iter):
        sessions = [generate_session(pi, rng) for _ in range(n_sessions)]
        es, ea, thr = select_elites(sessions, percentile)
        pi = update_policy(pi, es, ea, lam)
        log.append((np.mean([G for _, _, G in sessions]), thr))
    return pi, np.array(log)

t0 = time.time()
pi_cem, log = cem()
print(f'CEM: mean return per iteration (every 10th): {log[::10, 0].round(1)}   ({time.time() - t0:.0f} s)')
eps_cem = np.arange(1, len(log) + 1) * 250
R_q = np.mean([R for R, _ in runs['Q-learning']], axis=0)
plt.figure(figsize=(7, 4))
plt.plot(eps_cem, log[:, 0], 'o-', ms=3, label='CEM (mean return of the batch)')
plt.plot(np.arange(len(smooth(R_q))) + 50, smooth(R_q), label='Q-learning (from Q1.1)')
plt.xscale('log'); plt.xlabel('episodes played'); plt.ylabel('return'); plt.ylim(-800, 15)
plt.legend(); plt.grid(alpha=.3); plt.show()

**Watching the CEM policy.** The CEM policy is stochastic. Left: sampling actions from $\pi(a\mid s)$, as during training. Right: its most likely action $\arg\max_a\pi(a\mid s)$.

In [ ]:
rng_play = np.random.default_rng(2)
display(animate_episodes([('CEM, sampled', record_episode(lambda s: rng_play.choice(nA, p=pi_cem[s]), seed=7)),
                          ('CEM, most likely action', record_episode(lambda s: np.argmax(pi_cem[s]), seed=7))]))

How many episodes does each method need to reach a mean return of about 0? Why is CEM so much less data-efficient here? Name two things CEM throws away that Q-learning uses.

*Your answer:*

&nbsp;

### Q2.2 (3 pt). The percentile and the smoothing λ

In [ ]:
t0 = time.time()
for pct, lam in [(50, 0.5), (90, 0.5), (50, 1.0)]:
    _, lg = cem(percentile=pct, lam=lam)
    print(f'percentile {pct}, lambda {lam}: mean return at iterations 1, 10, 20, 40, 60: {lg[[0, 9, 19, 39, 59], 0].round(1)};'
          f'  threshold at 60: {lg[-1, 1]:.0f}')
print(f'({time.time() - t0:.0f} s)')

1. With percentile 90 the elite threshold is high (good elites!) but the mean return stays bad. Explain what goes wrong.
2. With λ = 1 (no smoothing) learning also stalls. What happens to a row of $\pi$ after a single update with λ = 1, and why can the method then never recover?

*Your answer:*

&nbsp;

## Part 3. Theory (5 pt, 1 each)

**T1.** Show that Expected SARSA whose target uses the **greedy** policy is exactly Q-learning. To which function does Expected SARSA converge when its target uses the ε-greedy policy?

**T2.** A replay buffer stores transitions $(s,a,r,s',a')$ produced by *older* versions of the policy. Which of SARSA, Expected SARSA, Q-learning and CEM still learn the right thing when trained on such data? Justify from the targets.

**T3.** Taxi truncates every episode at 200 steps. What goes wrong if the truncation is treated like termination, i.e. $Y=r$ at the 200th step? (Hint: the state does not contain the time.)

**T4.** Derive the tabular CEM update $\hat\pi(a\mid s)=N(s,a)/N(s)$ by maximizing the log-likelihood $\sum_{(s_i,a_i)\in\text{Elite}}\log\pi(a_i\mid s_i)$ subject to $\sum_a\pi(a\mid s)=1$ for every $s$ (Lagrange multipliers).

**T5.** Would CEM work on FrozenLake?

*Your answers (T1–T5):*

&nbsp;

## Part 4. Planning versus learning (2 pt)

Everything above is **learning**: the agent only calls `env.reset()` and `env.step(a)` and sees one sampled transition at a time. **Planning** means computing a policy from a **model** of the environment, without playing. Gymnasium exposes Taxi's model:
`env.unwrapped.P[s][a]` is a list of tuples `(probability, next_state, reward, terminated)`; for Taxi each list has one tuple with probability 1.

In [ ]:
P = env.unwrapped.P
s0, _ = env.reset(seed=0)
print(f'state {s0} = {list(env.unwrapped.decode(s0))}')
for a, name in enumerate(['south', 'north', 'east', 'west', 'pickup', 'dropoff']):
    print(f'  P[{s0}][{a}] ({name:7s}) =', P[s0][a])

**Value iteration** repeats the Bellman optimality backup over **all** states and actions, using the model instead of samples:
$$
Q(s,a)=\sum_{s',r}p(s',r\mid s,a)\,\bigl[r+\gamma\,V(s')\bigr],\qquad V(s)=\max_aQ(s,a),
$$
with $V(s')$ replaced by 0 if the transition terminates. Compare with the Q-learning target: the same backup, but with the sum over $(s',r)$ replaced by the **one sample** the environment returned.

The code below runs value iteration to convergence, then compares the result with the Q-learning table from Q1.1: same greedy return? same actions?

In [ ]:
def value_iteration(gamma=0.99, tol=1e-8):
    V = np.zeros(nS); sweeps = 0
    while True:
        Q = np.zeros((nS, nA))
        for s in range(nS):
            for a in range(nA):
                # Bellman optimality backup: expected reward + discounted value of the next state, using the model
                Q[s, a] = sum(p * (r + gamma * (0.0 if term else V[s2])) for p, s2, r, term in P[s][a])
        sweeps += 1
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tol:
            return Q, sweeps
        V = V_new

t0 = time.time()
Q_star, sweeps = value_iteration()
print(f'value iteration: {sweeps} sweeps over all {nS}x{nA} pairs = {sweeps * nS * nA:,} model queries, '
      f'0 episodes played  ({time.time() - t0:.1f} s)')
print(f'greedy return of value iteration: {greedy_return(Q_star):.1f}')

_, Q_ql = runs['Q-learning'][0]                       # Q-learning from Q1.1, seed 0, 2000 episodes
start_states = sorted({env.reset(seed=12345 + k)[0] for k in range(50)})
gap = np.abs(Q_ql[start_states].max(1) - Q_star[start_states].max(1))
print(f'Q-learning after 2000 episodes: greedy return {greedy_return(Q_ql):.1f}, '
      f'|V_QL - V*| on the evaluation start states: mean {gap.mean():.2f}, max {gap.max():.2f}')
same = np.mean(Q_ql[start_states].argmax(1) == Q_star[start_states].argmax(1))
ties = np.mean([np.isclose(Q_star[s][Q_ql[s].argmax()], Q_star[s].max()) for s in start_states])
print(f'same greedy action as value iteration in {same:.0%} of these start states; '
      f'Q-learning\'s action is optimal (Q* ties: several routes are equally short) in {ties:.0%}')

In [ ]:
filmstrip('value iteration, greedy', record_episode(lambda s: np.argmax(Q_star[s]), seed=0), n=12)

**Check your answer to Q1.** The optimal (value-iteration) policy from your start state:

In [ ]:
seed = MY_SEED if isinstance(MY_SEED, int) else 0
ep_opt = record_episode(lambda s: np.argmax(Q_star[s]), seed=seed)
print('actions:', ' '.join(ACTION_NAMES[a] for a in ep_opt[1]))
filmstrip(f'optimal policy from seed {seed}', ep_opt, n=min(len(ep_opt[0]), 12))

### Q4 (2 pt). Planning versus learning
1. Value iteration played no episodes, yet found the same greedy return as Q-learning. What did it use instead, and why is that not available to a robot taxi in a real city?
2. Sort the algorithms of this course into three groups by what they need from the environment: (i) only `reset()` and `step()` from the start state; (ii) a simulator that can be **set to any state** and stepped from there (a *sample model*); (iii) the full table of probabilities `P` (a *distribution model*). Algorithms: SARSA, Q-learning, CEM, value iteration, minimax/alpha-beta, MCTS.
3. Taxi has 500 states, so sweeping all of them is cheap. Why does that stop working for chess or Atari, even though their rules (the model) are known exactly? What do AlphaZero and DQN do instead?

*Your answer:*

&nbsp;